# Module 3a — UNESCO Merge (Researchers per Million, SDG 9.5.2)

**Revision note:** the first version of this notebook tried UNESCO's Bulk Data Download zip
files directly, guessing a dated URL — it 404'd, because UNESCO has since moved that hosting
and restructured the R&D bulk files. Rather than keep guessing dated URLs, this version uses
`unesco_reader` — a maintained Python package that wraps UNESCO's **live UIS Data API**
directly (no zip files, no dated folders, no guessing). This is a genuinely more robust
approach, not a workaround.

**What we're adding:** `researchers_per_million` — SDG indicator 9.5.2, "Researchers (in
full-time equivalent) per million inhabitants." World Bank doesn't publish this; it already
covers R&D expenditure (%GDP) and tertiary enrollment, so this is the one real gap left in
the Education & Innovation block.

Install the package once, in your terminal (not in the notebook):

```
pip install unesco-reader
```

In [2]:
import os
from pathlib import Path

while not (Path("config").exists() and Path("scripts").exists()):
    os.chdir("..")
    if Path.cwd() == Path.cwd().parent:
        raise RuntimeError("Could not locate project root")

print("Working directory set to:", os.getcwd())


Working directory set to: C:\Users\AJAY\Desktop\startup_implementation


In [2]:
%pip install unesco-reader

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Stage 1 — discover the real indicator ID

Same principle as before: don't hardcode a guessed ID and hope. Pull UNESCO's live indicator
catalog and search it by text for "researcher" — whatever ID comes back *is* correct, by
definition, because it's read straight from their API's current catalog rather than assumed
from a URL fragment.

In [8]:
import unesco_reader as uis
import pandas as pd

try:
    all_indicators = uis.available_indicators()
    matches = all_indicators[
        all_indicators["name"].str.contains("researcher", case=False, na=False)
    ]
    print(f"{len(matches)} indicators matched 'researcher':\n")
    display(matches[["indicatorCode", "name", "theme"]])
except Exception as e:
    print(f"Could not reach the UIS API from this environment ({e.__class__.__name__}).")
    print("Run this cell where you have real internet access.")
    matches = pd.DataFrame()


11 indicators matched 'researcher':



,indicatorCode,name,theme
1930,FRESP.SP.TFTE.BUSENTSP,Female researchers as a percentage of total re...,SCIENCE_TECHNOLOGY_INNOVATION
1931,FRESP.SP.TFTE.GOVSP,Female researchers as a percentage of total re...,SCIENCE_TECHNOLOGY_INNOVATION
1932,FRESP.SP.TFTE.HIEDUSP,Female researchers as a percentage of total re...,SCIENCE_TECHNOLOGY_INNOVATION
1933,FRESP.SP.TFTE.PRNPSP,Female researchers as a percentage of total re...,SCIENCE_TECHNOLOGY_INNOVATION
1934,FRESP.SP.THC.BUSENTSP,Female researchers as a percentage of total re...,SCIENCE_TECHNOLOGY_INNOVATION
1935,FRESP.SP.THC.GOVSP,Female researchers as a percentage of total re...,SCIENCE_TECHNOLOGY_INNOVATION
1936,FRESP.SP.THC.HIEDUSP,Female researchers as a percentage of total re...,SCIENCE_TECHNOLOGY_INNOVATION
1937,FRESP.SP.THC.PRNPSP,Female researchers as a percentage of total re...,SCIENCE_TECHNOLOGY_INNOVATION
1938,FRESP.TFTE,Researchers (FTE) - % Female,SCIENCE_TECHNOLOGY_INNOVATION
1939,FRESP.THC,Researchers (HC) - % Female,SCIENCE_TECHNOLOGY_INNOVATION


**Action needed:** from the matches above, find the row for "Researchers (in full-time
equivalent) per million inhabitants" (SDG 9.5.2 — *not* headcount, *not* by field/sector,
*not* gender-disaggregated) and set `indicator_id` below to its real `indicator_id`.

In [13]:
indicator_id = "RESDEN.INHAB.TFTE"  # <- set this from Stage 1's output, e.g. indicator_id = "24144"

if indicator_id is None and not matches.empty:
    print("Set indicator_id above using one of the IDs listed in Stage 1 before continuing.")


## Stage 2 — pull the data and merge

Once `indicator_id` is set, this pulls the full 2005–2025 series for every country, reshapes
it, and left-joins it onto `world_bank_wide.csv` — existing rows are untouched, this just
fills in one new column wherever UNESCO has coverage (real gaps expected; R&D surveys aren't
annual or universal across 217 countries).

In [14]:
if indicator_id:
    raw = uis.get_data(indicator=indicator_id, start=2005, end=2025)
    print("Raw shape:", raw.shape)
    print(raw.columns.tolist())
    display(raw.head())


Raw shape: (2448, 6)
['indicatorId', 'geoUnit', 'year', 'value', 'magnitude', 'qualifier']


,indicatorId,geoUnit,year,value,magnitude,qualifier
0,RESDEN.INHAB.TFTE,AGO,2011,48.42677,None,None
1,RESDEN.INHAB.TFTE,AGO,2016,18.93579,None,None
2,RESDEN.INHAB.TFTE,ALB,2008,155.23568,None,None
3,RESDEN.INHAB.TFTE,ALB,2021,449.10430,None,None
4,RESDEN.INHAB.TFTE,ALB,2022,428.69994,None,None


In [15]:
if indicator_id and not raw.empty:
    # geoUnit is UNESCO's own entity code — for national entities this is the ISO3 code,
    # matching our country_code column directly.
    unesco_df = raw.rename(columns={
        "geoUnit": "country_code",
        "year": "year",
        "value": "researchers_per_million",
    })[["country_code", "year", "researchers_per_million"]]

    unesco_df = unesco_df.dropna(subset=["researchers_per_million"])
    print(f"UNESCO rows: {len(unesco_df)}, countries: {unesco_df.country_code.nunique()}")

    wide_df = pd.read_csv("data/cleaned/world_bank_wide.csv")
    print("Before merge:", wide_df.shape)

    merged = wide_df.merge(unesco_df, on=["country_code", "year"], how="left")
    print("After merge:", merged.shape)
    coverage = merged["researchers_per_million"].notna().mean()
    print(f"Coverage: {coverage:.1%}")

    merged.to_csv("data/cleaned/world_bank_wide.csv", index=False)
    print("Saved back to data/cleaned/world_bank_wide.csv")


UNESCO rows: 2448, countries: 190
Before merge: (4557, 38)
After merge: (4557, 39)
Coverage: 32.9%
Saved back to data/cleaned/world_bank_wide.csv


## If `raw.columns` doesn't show `geoUnit`/`year`/`value`

UNESCO's API response shape has changed once already during this project (the WGI incident).
If Stage 2's column printout looks different from what the rename step expects, paste me the
actual `raw.columns.tolist()` and `raw.head()` output and I'll adjust the rename mapping —
don't guess and merge blind.

## Next

Once this runs cleanly, we move to WIPO (manual patents/trademarks export — still no public
API for this one, confirmed), then assemble everything into
`data/master/startup_master_dataset.csv`.

import pandas as pd
w = pd.read_csv("data/cleaned/world_bank_wide.csv")
print(w.shape)
print(w["researchers_per_million"].notna().sum())

In [16]:
import pandas as pd
w = pd.read_csv("data/cleaned/world_bank_wide.csv")
print(w.shape)
print(w["researchers_per_million"].notna().sum())

(4557, 39)
1498


In [20]:
import pandas as pd
patent = pd.read_csv("data/raw/wipo/patent_2005_2024.csv")
patent.head(10)

,Origin,Origin (Code),Office,Type,2005,2006,2007,2008,2009,2010,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
Afghanistan,AF,Total,Abroad,1.0,NaN,2.0,2.0,2.0,6.0,5.0,...,11.0,29.0,11.0,9.0,8.0,16.0,17.0,7.0,6.0,NaN
Afghanistan,AF,Total,Abroad (equivalent count),1.0,NaN,2.0,2.0,2.0,6.0,5.0,...,11.0,29.0,11.0,9.0,8.0,16.0,17.0,7.0,6.0,NaN
Albania,AL,Total,Total,1.0,NaN,NaN,NaN,4.0,NaN,5.0,...,37.0,18.0,21.0,12.0,NaN,32.0,28.0,28.0,NaN,NaN
Albania,AL,Total,Resident,NaN,NaN,NaN,NaN,NaN,NaN,3.0,...,22.0,16.0,15.0,6.0,NaN,26.0,24.0,17.0,NaN,NaN
Albania,AL,Total,Abroad,1.0,NaN,4.0,4.0,4.0,5.0,2.0,...,15.0,2.0,6.0,6.0,3.0,6.0,4.0,11.0,20.0,NaN
Albania,AL,Total,Total (equivalent count),1.0,NaN,NaN,NaN,4.0,NaN,5.0,...,51.0,18.0,21.0,24.0,NaN,40.0,35.0,71.0,NaN,NaN
Albania,AL,Total,Abroad (equivalent count),1.0,NaN,4.0,4.0,4.0,5.0,2.0,...,29.0,2.0,6.0,18.0,3.0,14.0,11.0,54.0,36.0,NaN
Algeria,DZ,Total,Total,65.0,59.0,88.0,NaN,NaN,80.0,102.0,...,112.0,158.0,162.0,119.0,170.0,280.0,483.0,1412.0,1117.0,NaN
Algeria,DZ,Total,Resident,59.0,58.0,84.0,NaN,NaN,76.0,94.0,...,106.0,149.0,152.0,113.0,163.0,268.0,471.0,1396.0,1084.0,NaN
Algeria,DZ,Total,Abroad,6.0,1.0,4.0,1.0,4.0,4.0,8.0,...,6.0,9.0,10.0,6.0,7.0,12.0,12.0,16.0,33.0,NaN
